
# IndiaKart E-Commerce Analytics — Phase 3: KPI Calculations

## 1. Project Overview

### Project Objective
Analyze IndiaKart's e-commerce data to calculate key business performance indicators (KPIs), evaluate performance against business targets, and identify opportunities for improvement.

### Tools Used
- MySQL Workbench — SQL queries and KPI calculations
- Python and Jupyter Notebook — documentation and presentation of results

### Database
`indiakart_analytics`

### Tables Used
- `orders da`
- `customers da`
- `order_items da`
- `payments da`
- `returns da`
- `inventory da`

### Analysis Period
24 June 2023 – 23 June 2025

### KPI Scope
This phase calculates the 10 KPIs specified in the IndiaKart Project PRD. Each KPI includes its business question, formula, SQL query, result, and business interpretation.



## 2. KPI Summary and Business Targets

This section summarizes the 10 key performance indicators defined in the IndiaKart Project PRD. The detailed calculations, SQL queries, results, and business interpretations are documented in the following sections.

| No. | KPI | Business Target |
|---:|---|---|
| 1 | Gross Merchandise Value (GMV) | Calculate total order value across all statuses |
| 2 | Net Revenue | Calculate final amount for Delivered orders |
| 3 | Average Order Value (AOV) | Net Revenue / Delivered Orders |
| 4 | Cancellation Rate | Below 10% |
| 5 | Return Rate | Below 8% |
| 6 | Customer Lifetime Value (CLV) | Premium segment should be 3x Regular |
| 7 | Month-over-Month (MoM) GMV Growth | Measure monthly growth or decline |
| 8 | Top Category Revenue Share | Concentration risk if one category exceeds 60% |
| 9 | Payment Failure Rate | Below 2% |
| 10 | Inventory Fill Rate | Above 95% |



## 3. KPI 1 — Gross Merchandise Value (GMV)

### Business Question
What is the total value of all orders placed on IndiaKart during the analysis period?

### Business Definition
GMV is the sum of `final_amount` across all order statuses, including Delivered, Cancelled, Returned, Shipped, and Processing orders, as specified in the project requirements.

### SQL Query
```sql
SELECT
    SUM(final_amount) AS total_gmv
FROM `orders da`;
```

### Result
- **Total GMV:** ₹3,147,385,544.58
- **Total GMV in crore:** ₹314.74 crore

### Business Interpretation
IndiaKart's total GMV across all order statuses is approximately ₹314.74 crore for the dataset's analysis period.

### Validation Note
This metric includes all order statuses according to the project definition. It represents total order value, not revenue from Delivered orders alone.



## 4. KPI 2 — Net Revenue

### Business Question
What is the total order value from orders that IndiaKart has successfully delivered?

### Business Definition
Net Revenue is calculated as the sum of `final_amount` for orders whose status is `Delivered`, according to the IndiaKart Project PRD.

### SQL Query
```sql
SELECT
    SUM(final_amount) AS net_revenue
FROM `orders da`
WHERE status = 'Delivered';
```

### Result
- **Net Revenue:** ₹2,053,858,611.96
- **Net Revenue in crore:** ₹205.39 crore

### Business Interpretation
IndiaKart recorded approximately ₹205.39 crore in final order amounts from Delivered orders during the analysis period.

### Comparison with GMV
GMV includes all order statuses, whereas this metric includes only Delivered orders. Therefore, Net Revenue is lower than total GMV.

### Validation Note
This KPI follows the PRD definition. It is the sum of final order amounts for Delivered orders, not a separately verified accounting or profit figure.



## 5. KPI 3 — Average Order Value (AOV)

### Business Question
How much does IndiaKart earn on average per Delivered order?

### Business Definition
Average Order Value (AOV) is calculated by dividing Net Revenue from Delivered orders by the total number of Delivered orders.

### Formula
AOV = Net Revenue / Number of Delivered Orders

### SQL Query
```sql
SELECT
    SUM(final_amount) / COUNT(*) AS average_order_value
FROM `orders da`
WHERE status = 'Delivered';
```

### Result
- **Average Order Value:** ₹63,197.59
- **Delivered Orders:** 32,499

### Business Interpretation
IndiaKart's average order value is approximately ₹63,197.59 per Delivered order. This metric helps the business understand the average monetary value of successfully delivered orders and track changes in customer purchasing behavior.

### Validation Note
The calculation includes only orders with status `Delivered`. The AOV is based on `final_amount`, consistent with the Net Revenue definition used in KPI 2.



## 6. KPI 4 — Order Cancellation Rate

### 1. Business Question
What percentage of IndiaKart orders are cancelled?

### 2. Business Definition
Cancellation Rate measures the percentage of total orders whose status is `Cancelled`.

### 3. Formula
Cancellation Rate (%) = (Cancelled Orders / Total Orders) × 100

### 4. SQL Query
```sql
SELECT
    COUNT(*) AS total_orders,
    SUM(
        CASE
            WHEN status = 'Cancelled' THEN 1
            ELSE 0
        END
    ) AS cancelled_orders,
    ROUND(
        SUM(
            CASE
                WHEN status = 'Cancelled' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS cancellation_rate_percent
FROM `orders da`;
```

### 5. Results
- **Total Orders:** 50,000
- **Cancelled Orders:** 5,996
- **Cancellation Rate:** 11.99%
- **Business Target:** Below 10%

### 6. Target Assessment
**Status: Target Not Met**

The cancellation rate of 11.99% is approximately 1.99 percentage points above the target of 10%.

### 7. Business Interpretation
Approximately 12 out of every 100 orders are cancelled. This may indicate opportunities to improve order confirmation, inventory availability, delivery estimates, and customer communication.

These possible causes should be investigated further before drawing a final conclusion.

### 8. Recommendation
Analyze cancellation rates by product category, state, payment method, and order date to identify where cancellations occur most frequently. Use those findings to prioritize corrective actions.

### 9. Validation Note
The calculation uses all 50,000 records in the orders table as the denominator and counts orders with status `Cancelled` as the numerator, following the project KPI definition.



## 7. KPI 5 — Order Return Rate

### 1. Business Question
What percentage of Delivered orders is represented by return records?

### 2. Business Definition
Return Rate is calculated by dividing the total number of records in the returns table by the number of Delivered orders, then multiplying by 100, as defined in the project requirements.

### 3. Formula
Return Rate (%) = (Total Return Records / Delivered Orders) × 100

### 4. SQL Query
```sql id="k8t2wq"
SELECT
    (SELECT COUNT(*)
     FROM `returns da`) AS total_return_records,

    (SELECT COUNT(*)
     FROM `orders da`
     WHERE status = 'Delivered') AS delivered_orders,

    ROUND(
        (SELECT COUNT(*) FROM `returns da`) * 100.0
        / (SELECT COUNT(*)
           FROM `orders da`
           WHERE status = 'Delivered'),
        2
    ) AS return_rate_percent;
```

### 5. Results
- **Total Return Records:** 10,000
- **Delivered Orders:** 32,499
- **Return Rate:** 30.77%
- **Business Target:** Below 8%

### 6. Target Assessment
**Status: Target Not Met**

The calculated rate of 30.77% is substantially above the target of 8%.

### 7. Business Interpretation
The result indicates a high return-record count relative to the number of Delivered orders. This warrants further investigation because returns can increase reverse-logistics costs and create additional work for customer service and warehouse teams.

### 8. Recommendation
Analyze return reasons, product categories, and return amounts to identify potential improvement areas. Prioritize the most frequent reasons for returns and investigate the products or categories contributing most to the problem.

### 9. Validation Note
This calculation follows the project-defined formula. It compares all records in the returns table with Delivered orders; it does not confirm that every return record belongs to a unique Delivered order. Therefore, treat 30.77% as the PRD-defined metric and validate order-level return matching before interpreting it as the true percentage of Delivered orders returned.



## 8. KPI 6 — Customer Lifetime Value (CLV)

### 1. Business Question
Which customer segment has the highest average lifetime spending, and does the Premium segment achieve three times the value of the Regular segment?

### 2. Business Definition
For this project, CLV is measured as the average `total_spent` per customer within each customer segment.

### 3. Formula
Average CLV = Sum of `total_spent` for a segment / Number of customers in that segment

Premium-to-Regular CLV Ratio = Premium Average CLV / Regular Average CLV

### 4. SQL Query
```sql
SELECT
    segment,
    ROUND(AVG(total_spent), 2)
        AS average_customer_lifetime_value
FROM `customers da`
GROUP BY segment
ORDER BY average_customer_lifetime_value DESC;
```

### 5. Results

| Customer Segment | Average CLV |
|---|---:|
| Premium | ₹532,735.00 |
| Regular | ₹361,870.72 |
| New | ₹274,866.48 |
| Budget | ₹186,178.81 |
| Inactive | ₹32,249.08 |

- **Premium-to-Regular CLV Ratio:** 1.47×
- **Business Target:** Premium CLV should be 3× Regular CLV.

### 6. Target Assessment
**Status: Target Not Met**

The Premium segment's average CLV is approximately 1.47 times the Regular segment's average CLV, below the target of 3 times.

### 7. Business Interpretation
Premium customers have the highest average lifetime spending, followed by Regular customers. However, the difference between these segments is smaller than the project's target requires.

### 8. Recommendation
Investigate purchase frequency, average order value, and repeat-purchase behavior across Premium and Regular customers. Consider targeted loyalty benefits and relevant product recommendations to encourage valuable repeat purchases.

### 9. Validation Note
The calculation uses the `total_spent` field in the customers table. Confirm how this field is defined and updated before treating it as independently verified lifetime revenue.


## 9. KPI 7 — Month-over-Month (MoM) GMV Growth

### 1. Business Question
How does IndiaKart's Gross Merchandise Value (GMV) change from one month to the next?

### 2. Business Definition
MoM GMV Growth measures the percentage increase or decrease in monthly GMV compared with the previous month.

### 3. Formula
MoM Growth (%) = ((Current Month GMV - Previous Month GMV) / Previous Month GMV) × 100

### 4. SQL Query

**Monthly GMV**

```sql
SELECT
    DATE_FORMAT(
        STR_TO_DATE(order_date, '%d-%m-%Y'),
        '%Y-%m'
    ) AS order_month,
    SUM(final_amount) AS monthly_gmv
FROM `orders da`
GROUP BY order_month
ORDER BY order_month;
```

**Monthly GMV Growth Calculation**

```sql
WITH monthly_gmv AS (
    SELECT
        DATE_FORMAT(
            STR_TO_DATE(order_date, '%d-%m-%Y'),
            '%Y-%m'
        ) AS order_month,
        SUM(final_amount) AS gmv
    FROM `orders da`
    GROUP BY order_month
)
SELECT
    order_month,
    ROUND(gmv, 2) AS monthly_gmv,
    ROUND(
        (gmv - LAG(gmv) OVER (ORDER BY order_month))
        * 100.0
        / NULLIF(
            LAG(gmv) OVER (ORDER BY order_month),
            0
        ),
        2
    ) AS mom_growth_percent
FROM monthly_gmv
ORDER BY order_month;
```

### 5. Key Results

| Month | Monthly GMV | MoM Growth |
|---|---:|---:|
| December 2024 | ₹13.31 crore | +5.41% |
| January 2025 | ₹13.79 crore | +3.64% |
| February 2025 | ₹11.41 crore | -17.26% |
| March 2025 | ₹13.65 crore | +19.61% |
| April 2025 | ₹13.08 crore | -4.15% |
| May 2025 | ₹13.58 crore | +3.84% |
| June 2025 | ₹10.36 crore | -23.74% |

### 6. Business Interpretation
GMV fluctuates from month to month. March 2025 recorded a 19.61% increase compared with February, while June 2025 showed a 23.74% decrease compared with May.

### 7. Recommendation
Investigate the factors behind the February decline and March recovery. Compare monthly order volume, category performance, cancellations, and payment failures to identify possible drivers of GMV changes.

### 8. Validation Note
The `LAG()` window function retrieves the previous month's GMV, and `NULLIF()` prevents division by zero. June 2023 and June 2025 are partial months in the dataset, so their growth figures should be interpreted cautiously. Monthly GMV includes all order statuses, consistent with the project's GMV definition.


## 10. KPI 8 — Top Category Revenue Share

### 1. Business Question
Which product category contributes the most revenue, and is IndiaKart overly dependent on one category?

### 2. Business Definition
Category Revenue Share measures the percentage of total order-item revenue contributed by each product category.

### 3. Formula
Category Revenue Share (%) = (Category Revenue / Total Order-Item Revenue) × 100

### 4. SQL Query
```sql
SELECT
    category,
    ROUND(SUM(total_price), 2) AS category_revenue,
    ROUND(
        SUM(total_price) * 100.0
        / (SELECT SUM(total_price)
           FROM `order_items da`),
        2
    ) AS revenue_share_percent
FROM `order_items da`
GROUP BY category
ORDER BY category_revenue DESC;
```

### 5. Key Results

| Product Category | Revenue | Revenue Share |
|---|---:|---:|
| Electronics | ₹164.24 crore | 57.41% |
| Sports & Fitness | ₹49.72 crore | 17.38% |
| Home & Kitchen | ₹24.43 crore | 8.54% |
| Fashion | ₹14.82 crore | 5.18% |
| Automotive | ₹12.35 crore | 4.32% |

### 6. Target Assessment
**Status: Within the Defined Threshold**

- Top category: Electronics
- Electronics revenue share: 57.41%
- Project threshold: Concentration risk if one category exceeds 60%

Electronics is below the 60% threshold, but its share is close enough to warrant monitoring.

### 7. Business Interpretation
Electronics is IndiaKart's largest product category, contributing 57.41% of total order-item revenue. Electronics and Sports & Fitness together account for 74.79%, indicating that a large share of revenue depends on these two categories.

### 8. Recommendation
Monitor Electronics revenue share regularly and explore opportunities to grow other categories, such as Home & Kitchen, Fashion, and Automotive. This may help reduce dependence on a small number of categories.

### 9. Validation Note
This calculation uses `order_items.total_price`, as specified for category-level revenue analysis. It is different from GMV, which is calculated using `orders.final_amount`. The two measures should not be treated as identical.



## 11. KPI 9 — Payment Failure Rate

### 1. Business Question
What percentage of payment attempts fail, and is IndiaKart meeting its payment reliability target?

### 2. Business Definition
Payment Failure Rate measures the percentage of payment records whose status is `Failed`.

### 3. Formula
Payment Failure Rate (%) = (Failed Payments / Total Payments) × 100

### 4. SQL Query
```sql
SELECT
    COUNT(*) AS total_payments,
    SUM(
        CASE
            WHEN status = 'Failed' THEN 1
            ELSE 0
        END
    ) AS failed_payments,
    ROUND(
        SUM(
            CASE
                WHEN status = 'Failed' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS payment_failure_rate_percent
FROM `payments da`;
```

### 5. Results
- **Total Payment Records:** 50,000
- **Failed Payment Records:** 1,748
- **Payment Failure Rate:** 3.50%
- **Business Target:** Below 2%

### 6. Target Assessment
**Status: Target Not Met**

The payment failure rate is 3.50%, which is 1.50 percentage points above the 2% target.

### 7. Business Interpretation
Approximately 3.5 out of every 100 payment records have a Failed status. Payment failures may interrupt purchases and create additional work for customer support.

### 8. Recommendation
Analyze failed payments by payment method, date, and order channel. Work with the payment service provider to investigate recurring failures and improve payment reliability.

### 9. Validation Note
This calculation uses the `status` column in the `payments da` table and counts payment records, not necessarily unique orders or customers. Further investigation is required to identify the causes of failures.



## 12. KPI 10 — Inventory Fill Rate

### 1. Business Question
What percentage of IndiaKart's product SKUs are currently marked as In Stock?

### 2. Business Definition
Inventory Fill Rate measures the percentage of total SKUs whose inventory status is `In Stock`.

### 3. Formula
Inventory Fill Rate (%) = (In-Stock SKUs / Total SKUs) × 100

### 4. SQL Query
```sql
SELECT
    COUNT(*) AS total_skus,
    SUM(
        CASE
            WHEN status = 'In Stock' THEN 1
            ELSE 0
        END
    ) AS in_stock_skus,
    ROUND(
        SUM(
            CASE
                WHEN status = 'In Stock' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS inventory_fill_rate_percent
FROM `inventory da`;
```

### 5. Results
- **Total SKUs:** 1,000
- **In-Stock SKUs:** 922
- **Low-Stock SKUs:** 76
- **Out-of-Stock SKUs:** 2
- **Inventory Fill Rate:** 92.20%
- **Business Target:** Above 95%

### 6. Target Assessment
**Status: Target Not Met**

The inventory fill rate is 92.20%, which is 2.80 percentage points below the 95% benchmark.

### 7. Business Interpretation
Of the 1,000 SKUs, 922 are marked In Stock, 76 are marked Low Stock, and 2 are marked Out of Stock. The results indicate that inventory availability needs attention to help reduce potential stock-related disruptions.

### 8. Recommendation
Prioritize replenishment for Low Stock and Out of Stock products. Review reorder levels, supplier lead times, and demand patterns to improve inventory availability.

### 9. Validation Note
This calculation follows the project definition: only SKUs with status `In Stock` count toward the fill rate. SKUs marked `Low Stock` are not included in the numerator.



# 13. Final KPI Summary and Business Assessment

This section summarizes the 10 KPIs calculated for the IndiaKart E-Commerce Analytics project. The results are compared with the business targets defined in the Project Requirements Document (PRD).

| No. | KPI | Calculated Result | Business Target | Assessment |
|---:|---|---:|---|---|
| 1 | Gross Merchandise Value (GMV) | ₹314.74 crore | Measure total order value | Calculated |
| 2 | Net Revenue | ₹205.39 crore | Measure Delivered order value | Calculated |
| 3 | Average Order Value (AOV) | ₹63,197.59 | Measure average Delivered order value | Calculated |
| 4 | Cancellation Rate | 11.99% | Below 10% | Not Met |
| 5 | Return Rate | 30.77% | Below 8% | Not Met — Validate order-level matching |
| 6 | Customer Lifetime Value (CLV) | Premium: ₹532,735.00; Regular: ₹361,870.72 | Premium CLV should be 3× Regular | Not Met — 1.47× |
| 7 | MoM GMV Growth | Varies by month | Monitor monthly growth | Calculated |
| 8 | Top Category Revenue Share | Electronics: 57.41% | Concentration risk if above 60% | Within Threshold |
| 9 | Payment Failure Rate | 3.50% | Below 2% | Not Met |
| 10 | Inventory Fill Rate | 92.20% | Above 95% | Not Met |

## Overall Business Findings

1. **Order cancellations need attention:** The cancellation rate is 11.99%, above the target of 10%.
2. **Returns require investigation:** The PRD-defined return rate is 30.77%. Order-level matching should be validated before interpreting this as the actual percentage of Delivered orders returned.
3. **Premium customer value is below the benchmark:** Premium customers have 1.47× the average lifetime spending of Regular customers, below the 3× target.
4. **Payment reliability needs improvement:** The payment failure rate is 3.50%, above the 2% target.
5. **Inventory availability is below target:** The inventory fill rate is 92.20%, below the 95% target.
6. **Electronics dominates category revenue:** Electronics contributes 57.41% of order-item revenue, below but relatively close to the 60% concentration threshold.

## Conclusion

The Phase 3 analysis provides a consolidated view of IndiaKart's revenue, customer value, order cancellations, returns, payment reliability, category performance, and inventory availability.

The results identify several areas for further investigation and improvement. These findings can guide the design of the Phase 4 dashboard and the final management report.

## Important Methodology Notes

- GMV includes all order statuses, as defined in the PRD.
- Net Revenue and AOV use Delivered orders only.
- Category revenue share uses `order_items.total_price`, whereas GMV uses `orders.final_amount`.
- Return Rate follows the PRD formula but requires validation of order-level matching.
- June 2023 and June 2025 are partial months, so their MoM growth figures should be interpreted cautiously.



# 14. Bonus Analysis — Customer Cohort Analysis

##  Objective

The objective of cohort analysis is to understand customer purchasing behavior after registration. Customers are grouped by their registration month, and their order activity is tracked during the first three months.

##  Business Questions

1. How many orders do customers place in their registration month?
2. How many orders do they place in the following month?
3. How many orders do they place in the third month?
4. Does purchasing activity continue after customer registration?

##  Methodology

- **Cohort:** The month in which a customer registered.
- **Month 1:** The customer's registration month.
- **Month 2:** The month immediately after registration.
- **Month 3:** The second month after registration.
- **Order activity:** The number of orders placed by customers belonging to each cohort.

##  Tools Used

- Python
- Pandas
- Jupyter Notebook


In [3]:

import pandas as pd
from pathlib import Path

# Locate the main project folder
project_folder = Path.cwd().parent

# Load the customer and order datasets
customers = pd.read_csv(project_folder / "customers DA.csv")
orders = pd.read_csv(project_folder / "orders DA.csv")

# Confirm that the data loaded correctly
print("Customers shape:", customers.shape)
print("Orders shape:", orders.shape)

print("\nCustomer columns:")
print(customers.columns.tolist())

print("\nOrder columns:")
print(orders.columns.tolist())


Customers shape: (10000, 17)
Orders shape: (50000, 19)

Customer columns:
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'state', 'pincode', 'gender', 'age', 'segment', 'registration_date', 'last_login_date', 'is_verified', 'is_active', 'total_orders', 'total_spent']

Order columns:
['order_id', 'customer_id', 'order_date', 'order_time', 'status', 'city', 'state', 'pincode', 'total_amount', 'gst_amount', 'shipping_charge', 'discount_amount', 'final_amount', 'payment_method', 'shipping_partner', 'tracking_id', 'delivered_date', 'is_cod', 'channel']


In [4]:

import pandas as pd

# Create working copies so the original data remains unchanged
customers_cohort = customers.copy()
orders_cohort = orders.copy()

# Convert registration dates into datetime format
customers_cohort["registration_date"] = pd.to_datetime(
    customers_cohort["registration_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# Convert order dates into datetime format
orders_cohort["order_date"] = pd.to_datetime(
    orders_cohort["order_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# Check the prepared data
print("Customers:", customers_cohort.shape)
print("Orders:", orders_cohort.shape)

print("\nMissing registration dates:",
      customers_cohort["registration_date"].isna().sum())

print("Missing order dates:",
      orders_cohort["order_date"].isna().sum())

print("\nCustomer registration date sample:")
print(customers_cohort[["customer_id", "registration_date"]].head())

print("\nOrder date sample:")
print(orders_cohort[["order_id", "customer_id", "order_date"]].head())


Customers: (10000, 17)
Orders: (50000, 19)

Missing registration dates: 0
Missing order dates: 0

Customer registration date sample:
  customer_id registration_date
0   CUST00001        2024-07-30
1   CUST00002        2023-07-05
2   CUST00003        2022-05-14
3   CUST00004        2025-01-21
4   CUST00005        2023-06-06

Order date sample:
    order_id customer_id order_date
0  ORD000001   CUST02946 2025-05-22
1  ORD000002   CUST04232 2023-10-16
2  ORD000003   CUST08078 2023-10-08
3  ORD000004   CUST09612 2023-10-02
4  ORD000005   CUST03065 2024-06-15


In [7]:

# Assign each customer to their registration-month cohort
customers_cohort["cohort_month"] = (
    customers_cohort["registration_date"]
    .dt.to_period("M")
)

# Assign each order to the month it was placed
orders_cohort["order_month"] = (
    orders_cohort["order_date"]
    .dt.to_period("M")
)

# Display sample results
print("Customer Cohort Sample:")
print(
    customers_cohort[
        ["customer_id", "registration_date", "cohort_month"]
    ].head(10)
)

print("\nOrder Month Sample:")
print(
    orders_cohort[
        ["order_id", "customer_id", "order_date", "order_month"]
    ].head(10)
)


Customer Cohort Sample:
  customer_id registration_date cohort_month
0   CUST00001        2024-07-30      2024-07
1   CUST00002        2023-07-05      2023-07
2   CUST00003        2022-05-14      2022-05
3   CUST00004        2025-01-21      2025-01
4   CUST00005        2023-06-06      2023-06
5   CUST00006        2022-04-06      2022-04
6   CUST00007        2022-06-10      2022-06
7   CUST00008        2021-08-04      2021-08
8   CUST00009        2022-11-24      2022-11
9   CUST00010        2024-11-20      2024-11

Order Month Sample:
    order_id customer_id order_date order_month
0  ORD000001   CUST02946 2025-05-22     2025-05
1  ORD000002   CUST04232 2023-10-16     2023-10
2  ORD000003   CUST08078 2023-10-08     2023-10
3  ORD000004   CUST09612 2023-10-02     2023-10
4  ORD000005   CUST03065 2024-06-15     2024-06
5  ORD000006   CUST00643 2025-03-02     2025-03
6  ORD000007   CUST00754 2025-04-12     2025-04
7  ORD000008   CUST02760 2024-08-12     2024-08
8  ORD000009   CUST01691 202

In [8]:

# Connect each order to the customer's registration cohort
cohort_orders = orders_cohort.merge(
    customers_cohort[["customer_id", "cohort_month"]],
    on="customer_id",
    how="left"
)

# Calculate how many months after registration each order occurred
cohort_orders["months_since_registration"] = (
    (cohort_orders["order_month"].dt.year
     - cohort_orders["cohort_month"].dt.year) * 12
    + (cohort_orders["order_month"].dt.month
       - cohort_orders["cohort_month"].dt.month)
)

# Keep orders placed during the registration month
# and the following two months
cohort_orders_3m = cohort_orders[
    cohort_orders["months_since_registration"].between(0, 2)
].copy()

# Count orders for each cohort and month
cohort_order_counts = (
    cohort_orders_3m
    .groupby(
        ["cohort_month", "months_since_registration"]
    )["order_id"]
    .nunique()
    .unstack(fill_value=0)
)

# Give the columns clear business-friendly names
cohort_order_counts = cohort_order_counts.rename(
    columns={
        0: "Month 1 Orders",
        1: "Month 2 Orders",
        2: "Month 3 Orders"
    }
)

# Ensure all three month columns exist
cohort_order_counts = cohort_order_counts.reindex(
    columns=[
        "Month 1 Orders",
        "Month 2 Orders",
        "Month 3 Orders"
    ],
    fill_value=0
)

# Display the cohort table
print("Cohort Order Counts — First Three Months")
display(cohort_order_counts)


Cohort Order Counts — First Three Months


months_since_registration,Month 1 Orders,Month 2 Orders,Month 3 Orders
cohort_month,,,
2023-04,0,0,6
2023-05,0,6,37
2023-06,5,32,28
2023-07,34,40,33
2023-08,30,30,36
2023-09,29,40,46
2023-10,36,35,31
2023-11,39,31,30
2023-12,41,23,38


In [10]:

# Step 1: Count the total registered customers in each cohort
cohort_sizes = (
    customers_cohort.groupby("cohort_month")["customer_id"]
    .nunique()
    .rename("Cohort Size")
)

# Step 2: Count unique customers who placed orders
# in each of the first three months after registration
cohort_customer_counts = (
    cohort_orders_3m
    .groupby(["cohort_month", "months_since_registration"])["customer_id"]
    .nunique()
    .unstack(fill_value=0)
)

# Step 3: Ensure Month 1, Month 2 and Month 3 columns exist
cohort_customer_counts = cohort_customer_counts.reindex(
    columns=[0, 1, 2],
    fill_value=0
)

cohort_customer_counts.columns = [
    "Month 1 Customers",
    "Month 2 Customers",
    "Month 3 Customers"
]

# Step 4: Add each cohort's total registered customer count
cohort_customer_counts = cohort_sizes.to_frame().join(
    cohort_customer_counts,
    how="left"
)

# Fill missing order counts with zero
month_columns = [
    "Month 1 Customers",
    "Month 2 Customers",
    "Month 3 Customers"
]

cohort_customer_counts[month_columns] = (
    cohort_customer_counts[month_columns]
    .fillna(0)
    .astype(int)
)

# Step 5: Calculate monthly order activity as a percentage
# of the original registered customer cohort
cohort_retention = (
    cohort_customer_counts[month_columns]
    .div(cohort_customer_counts["Cohort Size"], axis=0)
    * 100
).round(2)

# Step 6: Display both tables
print("Customer Cohort Sizes and Monthly Order Activity")
display(cohort_customer_counts)

print("Customer Cohort Retention (%)")
display(cohort_retention)


Customer Cohort Sizes and Monthly Order Activity


,Cohort Size,Month 1 Customers,Month 2 Customers,Month 3 Customers
cohort_month,,,,
2020-06,29,0,0,0
2020-07,182,0,0,0
2020-08,193,0,0,0
2020-09,127,0,0,0
2020-10,182,0,0,0
...,...,...,...,...
2025-02,163,21,27,29
2025-03,180,32,35,32
2025-04,159,35,26,25


Customer Cohort Retention (%)


,Month 1 Customers,Month 2 Customers,Month 3 Customers
cohort_month,,,
2020-06,0.00,0.00,0.00
2020-07,0.00,0.00,0.00
2020-08,0.00,0.00,0.00
2020-09,0.00,0.00,0.00
2020-10,0.00,0.00,0.00
...,...,...,...
2025-02,12.88,16.56,17.79
2025-03,17.78,19.44,17.78
2025-04,22.01,16.35,15.72



## 14.1 Cohort Analysis — Results and Business Insights

### Objective
To understand customer ordering activity during the first three months after registration by grouping customers according to their registration month.

### Key Findings

1. **Month 1 Customer Activity:** For the February–April 2025 cohorts, approximately 12.88%–22.01% of registered customers placed at least one order during their registration month.

2. **Changes in Customer Activity:** For the April 2025 cohort, ordering activity decreased from 22.01% in Month 1 to 16.35% in Month 2 and 15.72% in Month 3.

3. **Recent Cohorts Have Incomplete Data:** The May and June 2025 cohorts do not have complete three-month observation periods because the dataset ends in June 2025. Their unavailable future months should not be interpreted as confirmed zero activity.

4. **Customer Engagement Opportunity:** IndiaKart can investigate declining ordering activity and test personalised offers, follow-up campaigns, and post-purchase engagement strategies to encourage repeat purchases.

### Methodology

- Customers were grouped by their registration month (`cohort_month`).
- Orders were matched to customers using `customer_id`.
- Customer activity was calculated for Month 1, Month 2, and Month 3 after registration.
- Monthly activity percentages were calculated using the total number of registered customers in each cohort as the denominator.

### Limitations

- This analysis measures the percentage of registered customers who placed at least one order in each relative month.
- It does not measure strict month-to-month retention because customers do not need to have ordered in the previous month.
- Recent cohorts have incomplete observation periods, so future months must be treated as unavailable rather than confirmed zero activity.

### Business Recommendations

1. Monitor customer ordering activity across registration cohorts.
2. Investigate why ordering activity declines for some cohorts.
3. Test targeted engagement campaigns and personalised offers to encourage repeat purchases.
4. Recalculate cohort metrics when additional months of order data become available.

### Conclusion

Cohort analysis provides IndiaKart with a way to monitor customer ordering behaviour after registration and identify opportunities to improve customer engagement and repeat purchasing.
